In [ ]:
import mlplant

In [ ]:
@mlplant.config
def _():
    LEARNING_RATE = 0.01
    N_ESTIMATORS = 100
    TEST_SIZE = 0.2
    RANDOM_STATE = 42

In [ ]:
@mlplant.load_data
def _():
    import pandas as pd
    df = pd.read_csv('data.csv')

In [ ]:
@mlplant.preprocessing
def _():
    df = df.dropna()
    df = df.drop_duplicates()

In [ ]:
@mlplant.features
def _():
    import matplotlib.pyplot as plt

    X = df.drop(columns=['target'])
    y = df['target']

    # noise below — will be stripped by mlplant
    plt.hist(y)
    plt.show()
    print(X.head())

In [ ]:
@mlplant.features
def _():
    X['ratio'] = X['col_a'] / (X['col_b'] + 1e-8)
    X['log_col_a'] = X['col_a'].apply(lambda v: v ** 0.5)

In [ ]:
@mlplant.train
def _():
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    model = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)

In [ ]:
@mlplant.evaluate
def _():
    from sklearn.metrics import accuracy_score

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

In [ ]:
@mlplant.artifacts
def _():
    import joblib
    joblib.dump(model, 'model.joblib')

In [ ]:
@mlplant.predict
def _():
    import joblib
    import pandas as pd

    _model = joblib.load('model.joblib')

    def predict(data: dict):
        df = pd.DataFrame([data])
        return int(_model.predict(df)[0])